In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

col_names = [
    "duration", "protocol_type", "service", "flag", "src_bytes",
    "dst_bytes", "land", "wrong_fragment", "urgent", "hot",
    "num_failed_logins", "logged_in", "num_compromised", "root_shell",
    "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate", "label", "difficulty"
]

train_df = pd.read_csv("../data/KDDTrain+.txt", names=col_names)
test_df  = pd.read_csv("../data/KDDTest+.txt",  names=col_names)

# Label simplification
train_df["label"] = train_df["label"].apply(
    lambda x: "normal" if x == "normal" else "attack"
)
test_df["label"] = test_df["label"].apply(
    lambda x: "normal" if x == "normal" else "attack"
)

# Encode categorical columns
le = LabelEncoder()
categorical_cols = ["protocol_type", "service", "flag"]
for col in categorical_cols:
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col]  = le.fit_transform(test_df[col])

# Encode labels
train_df["label_encoded"] = le.fit_transform(train_df["label"])
test_df["label_encoded"]  = le.transform(test_df["label"])

# Separate features and labels
features_to_drop = ["label", "difficulty", "label_encoded"]
X_train = train_df.drop(columns=features_to_drop)
y_train = train_df["label_encoded"]
X_test  = test_df.drop(columns=features_to_drop)
y_test  = test_df["label_encoded"]

print("Data ready.")
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")


Data ready.
X_train: (125973, 41) | X_test: (22544, 41)


In [2]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Scaling complete.")

Scaling complete.


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
import time

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100,
                                             random_state=42,
                                             n_jobs=-1),
    "KNN":           KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "SVM":           SVC(kernel="rbf", random_state=42),
    "XGBoost":       XGBClassifier(n_estimators=100,
                                    random_state=42,
                                    n_jobs=-1,
                                    eval_metric="logloss",
                                    verbosity=0)
}

trained_models = {}

for name, model in models.items():
    print(f"Training {name}...", end=" ", flush=True)
    start = time.time()
    model.fit(X_train_scaled, y_train)
    elapsed = time.time() - start
    trained_models[name] = model
    print(f"done in {elapsed:.1f}s")

print("\nAll models trained successfully.")


Training Decision Tree... done in 2.9s
Training Random Forest... done in 14.8s
Training KNN... done in 0.1s
Training SVM... done in 140.4s
Training XGBoost... done in 2.9s

All models trained successfully.


In [4]:
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score)

results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    results.append({
        "Model":     name,
        "Accuracy":  round(accuracy_score(y_test, y_pred) * 100, 2),
        "Precision": round(precision_score(y_test, y_pred) * 100, 2),
        "Recall":    round(recall_score(y_test, y_pred) * 100, 2),
        "F1 Score":  round(f1_score(y_test, y_pred) * 100, 2)
    })

results_df = pd.DataFrame(results).sort_values("F1 Score", ascending=False)
results_df = results_df.reset_index(drop=True)

print(results_df.to_string(index=False))

results_df.to_csv("../outputs/model_results.csv", index=False)
print("\nResults saved to outputs/model_results.csv")


        Model  Accuracy  Precision  Recall  F1 Score
      XGBoost     80.40      69.54   96.99     81.00
Decision Tree     79.04      67.97   97.12     79.97
          SVM     78.14      66.76   98.07     79.45
Random Forest     77.34      66.14   97.14     78.69
          KNN     76.57      65.24   97.62     78.21

Results saved to outputs/model_results.csv
